<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/01_prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 - Prompt Chaining for a Customer Support AI

**Cash Johnson | BUS4 118S - Agentic AI for Business | Prof. Haubrich**

**Tools used:** Google Colab + Google Gemini API (`google-genai` SDK, model `gemini-3.6-flash`).
No orchestration framework. LangChain and LangBase both do prompt chaining for you, and that is
the problem for this exercise: the dependency between steps is the thing being graded, so hiding it
inside `SequentialChain` would hide the assignment. Every handoff here is a visible Python variable.

**Goal:** a four-step prompt chain that runs a support ticket end to end, where each step's output is
the next step's input. Nothing is pasted by hand between steps.

**Scenario:** VNTG OS, the two-sided vintage consignment app I built for BUS4 110B. Two sides write
in: consignors who dropped off clothing, and buyers who bought it.

---

### Chain map

| Step | Name | Input | Output | Consumed by |
|---|---|---|---|---|
| 1 | Classify | raw ticket text | JSON: `category`, `side`, `urgency`, `missing_info[]`, `payout_at_risk_usd` | steps 2, 3, 4 |
| 2 | Gather | step 1 `missing_info[]` | the exact clarifying questions, one per missing field | step 3 |
| 3 | Resolve | step 1 JSON + answers + **policy retrieved by step 1's category** | resolution + reply draft | step 4 |
| 4 | Route | step 1 + step 3 + ticket | deterministic escalation rule, human handoff note if it fires | agent inbox |

### Techniques from the module used in this notebook

| Module technique | Where it shows up here |
|---|---|
| **Prompt chaining** | the whole notebook: 4 steps, each consuming the last |
| **Context-Aware Decomposition (CAD)** | one support ticket broken into classify / gather / resolve / route, each step aware of the overall goal |
| **System prompt vs user prompt** | every step has both, labeled, with the role and constraints in the system prompt |
| **Role-based prompting** | "intake classifier", "senior support specialist", "internal handoff writer" |
| **Clarity and specificity** | literal JSON schema and enums in step 1 instead of "tell me what the issue is" |
| **Content structuring** | every step specifies its output format, section by section |
| **Negative prompting** | "do not invent details", "do not cite a policy rule not in the block", "no greeting, no sign-off" |
| **Few-shot prompting with instruction** | step 2, tested head to head against the zero-shot version |
| **A/B testing** | section 5: two versions of step 2, same input, measured on three checks, winner used downstream |
| **Including contextual data (RAG-style)** | step 3 retrieves only the policy lines matching step 1's category, not the whole policy |
| **Temperature** | 0 on every classification step so the same ticket routes the same way twice |
| **Learning from failed prompts** | section 9: four failures and what each one changed |

## 0. Setup

Free-tier Gemini API key from https://aistudio.google.com/apikey.
In Colab: **key icon in the left sidebar > Add new secret > name it `GOOGLE_API_KEY` > toggle
notebook access on.** If the secret is not found, the cell falls back to a hidden prompt.

In [46]:
!pip install -q -U google-genai

In [47]:
from google import genai
from google.genai import types
import json

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.6-flash'

# Confirm the model name is live on this key before the chain runs.
available = [m.name for m in client.models.list() if 'generateContent' in (m.supported_actions or [])]
assert any(m.split('/')[-1] == MODEL for m in available), f'{MODEL} not available. Options: {available[:10]}'
print('Connected. Using:', MODEL)

Connected. Using: gemini-3.6-flash


In [48]:
def ask(user_prompt, system=None, temperature=0.0, json_mode=False, model=MODEL):
    """One call to Gemini.

    system      -> the SYSTEM prompt: role, constraints, rules of engagement
    user_prompt -> the USER prompt: the task and the data for this specific call
    temperature -> 0 by default, so the chain routes the same ticket the same way every run
    """
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    if json_mode:
        kwargs['response_mime_type'] = 'application/json'
    resp = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(**kwargs),
    )
    return (resp.text or '').strip()

print('helper ready')

helper ready


## 1. The inputs

Two tickets. Ticket A is the walkthrough. Ticket B runs through the same chain at the end to prove
the escalation rule actually fires instead of just existing in a comment.

In [49]:
TICKET_A = """Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus"""

TICKET_B = """Subject: CHARGED TWICE

I ordered the Ralph Lauren rugby ($480) on Tuesday and my bank shows two charges for the
same amount 4 minutes apart. I have already called my bank about disputing it. I need this
fixed today or I am filing the chargeback.

Danielle R."""

print(TICKET_A)

Subject: still waiting??

hey so i dropped a bunch of stuff off at the store like 3 weeks ago and nothing has
shown up on my account yet. one of them was a carhartt detroit jacket that should be
worth a lot. also my last payout was way less than i expected. can someone look at this

- marcus


## 2. The policy base, indexed by category

**Technique: including contextual data (RAG-style grounding).**

The model is never asked what the policy is, because it does not know and will happily invent one.
Policy is passed in as context. The retrieval step is keyed off `category`, which step 1 produced,
so step 3 sees only the lines that apply to this ticket plus the rules that always apply. Less
irrelevant context in the window, and one more real dependency in the chain.

In [50]:
POLICY_INDEX = {
    'intake_delay': [
        'Dropped-off items are photographed, priced, and listed within 10 business days.',
        'Drop-off volume spikes may extend intake; the SLA does not pause, it is missed.',
        'A consignor may ask for the current status of any item by drop-off date.',
    ],
    'payout_dispute': [
        'Split: consignor 60% / store 40% of final sale price.',
        'Items sold at $200 or more pay the consignor 65%.',
        'Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.',
        'A return reverses that item\u2019s payout line on the next cycle.',
        'Never promise a specific payout amount before the return window closes.',
    ],
    'billing_error': [
        'Duplicate charges are verified against the payment processor before any refund.',
        'Confirmed duplicates are refunded to the original method within 5 business days.',
        'Support never confirms a refund before the processor record is checked.',
    ],
    'item_condition': [
        'Buyers may return within 7 days for condition not matching the listing.',
        'Measurements are entered at intake and are not independently verified.',
        'A consignor may request one re-list at a new price per item.',
    ],
    'shipping': [
        'Local pickup and standard shipping only; no expedited service.',
        'Tracking is emailed at the time the label is created.',
    ],
    'account_access': [
        'Account recovery requires the email on file; support cannot change it over chat.',
    ],
    'other': [],
}

ALWAYS_APPLIES = [
    'Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, '
    'any mention of a chargeback or bank dispute, or urgency = high.',
    'Do not promise timelines that are not written in policy.',
]

def retrieve_policy(category: str) -> str:
    """Return only the policy lines relevant to this ticket's category, plus the always-on rules."""
    lines = POLICY_INDEX.get(category, []) + ALWAYS_APPLIES
    return '\n'.join(f'- {l}' for l in lines)

TOTAL_LINES = sum(len(v) for v in POLICY_INDEX.values()) + len(ALWAYS_APPLIES)
print(f'{TOTAL_LINES} policy lines in the base\n')
print('retrieve_policy("payout_dispute") returns:\n')
print(retrieve_policy('payout_dispute'))

19 policy lines in the base

retrieve_policy("payout_dispute") returns:

- Split: consignor 60% / store 40% of final sale price.
- Items sold at $200 or more pay the consignor 65%.
- Payouts run the 1st and the 15th, only on items past the 7-day buyer return window.
- A return reverses that item’s payout line on the next cycle.
- Never promise a specific payout amount before the return window closes.
- Escalate to a human on any of: a payout dispute over $150, a suspected duplicate charge, any mention of a chargeback or bank dispute, or urgency = high.
- Do not promise timelines that are not written in policy.


## 3. Step 1 v1 - the version that broke the chain

**Technique: learning from failed prompts.**

First attempt. It reads like a reasonable instruction, and the answer it returns is not wrong, it is
just unparseable. Step 2 needs to loop over a list of missing fields, and prose is not a list.

In [51]:
# USER prompt. No system prompt at all in v1 - that is part of why it drifts.
STEP1_V1 = """You are a customer support assistant for a vintage consignment store.
Read the ticket and tell me what the issue is, how urgent it is, and what information is missing.

TICKET:
{ticket}"""
v1_out = ask(STEP1_V1.format(ticket=TICKET_A))
print(v1_out)

Here is the breakdown of the ticket:

### 1. What the issue is
Marcus has two distinct issues:
* **Unprocessed Drop-Off / Missing Inventory:** He dropped off items 3 weeks ago (including a high-value Carhartt Detroit jacket) that have not yet been logged or displayed on his consignor account. 
* **Payout Discrepancy:** His most recent payout was significantly lower than he anticipated, and he needs a clarification or review of his earnings.

### 2. Urgency Level
**Medium to High**
* **Why:** While this isn't a critical emergency, it has been 3 weeks since his drop-off, which is a long wait time. Additionally, high-value items and financial/payout concerns tend to cause customer anxiety and frustration quickly. Addressing this promptly will prevent the situation from escalating into a lost customer or a potential dispute.

### 3. Missing Information
To resolve Marcus's issue, we need to request the following details from him:
* **Account Identifier:** His Consignor Account ID, or the fu

In [52]:
# Proof of the failure, not a claim about it.
try:
    parsed = json.loads(v1_out)
    print('parsed ok')
except json.JSONDecodeError as e:
    print('STEP 2 CANNOT CONSUME THIS ->', type(e).__name__, ':', e)

STEP 2 CANNOT CONSUME THIS -> JSONDecodeError : Expecting value: line 1 column 1 (char 0)


## 4. Step 1 v2 - classify

What changed, and why:

| Change | Technique | Reason |
|---|---|---|
| Role and scope moved into a system prompt | system prompt optimization | keeps the task frame out of the user turn, so ticket text cannot overwrite it |
| Explicit JSON schema, field by field | clarity and specificity | step 2 indexes `missing_info`, so the key name is a contract |
| Enums on `category`, `side`, `urgency` | content structuring | free-text labels made the step 4 routing rule unwriteable, and `category` is now the retrieval key |
| `payout_at_risk_usd` as a number or null | clarity and specificity | step 4 compares it to a dollar threshold |
| `response_mime_type='application/json'` | constraint at the API level | forces valid JSON instead of asking politely |
| "Do not invent details not present in the ticket" | negative prompting | v1 guessed a drop-off date that was never stated |
| temperature 0 | temperature control | same ticket, same classification, every run |

In [53]:
# SYSTEM prompt - role, scope, and what this step must never do.
STEP1_SYSTEM = """You are the intake classifier for VNTG OS, a two-sided vintage consignment
marketplace. You classify inbound support tickets. You do not answer the customer, you do not
apologize, and you do not propose a fix. Output JSON only."""

# USER prompt - the task and the data.
STEP1_V2 = """Classify the support ticket below.

Return ONLY a JSON object with exactly these keys:
  "category": one of ["intake_delay", "payout_dispute", "billing_error", "item_condition",
                      "shipping", "account_access", "other"]
  "side": one of ["consignor", "buyer", "unknown"]
  "urgency": one of ["low", "medium", "high"]
  "payout_at_risk_usd": a number, or null if no dollar amount is stated or implied
  "summary": one sentence, max 20 words, neutral tone
  "missing_info": an array of 2 to 5 short strings naming the exact fields support must have
                  before this ticket can be resolved

Rules:
- Use only facts stated in the ticket. Do not invent dates, order numbers, or amounts.
- If the customer raises two problems, classify the one with the higher financial impact.
- urgency = high only if money has already left the customer's account, a chargeback or bank
  dispute is mentioned, or the customer states a same-day deadline.

TICKET:
{ticket}"""

step1_raw = ask(STEP1_V2.format(ticket=TICKET_A), system=STEP1_SYSTEM, json_mode=True)
step1 = json.loads(step1_raw)   # parses now
print(json.dumps(step1, indent=2))

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 5. Step 2 - gather the missing information, as an A/B test

**Technique: A/B testing, and few-shot prompting with instruction.**

Same task, same input, two prompt versions, measured on the same three checks:

| Check | Pass condition |
|---|---|
| question count | exactly one numbered question per missing field |
| length | under 110 words |
| no support filler | none of "we apologize for the inconvenience", "thank you for reaching out", "we are sorry to hear" |

- **Version A (zero-shot):** instruction and constraints only.
- **Version B (few-shot with instruction):** identical instruction, plus three input/output example
  pairs showing the exact question style wanted.

Neither version sees the raw ticket. Both see only `step1['missing_info']` and `step1['side']`.
That is the dependency: change step 1 and the questions change with it.

In [ ]:
STEP2_SYSTEM = """You write short clarifying messages to customers of a vintage consignment
marketplace. Plain English, no corporate filler, no apology paragraph."""

# --- VERSION A: zero-shot, instruction only ---
STEP2_A = """A {side} has an open ticket. Support cannot resolve it without these fields:

{missing}

Write the reply that asks for them.
Constraints:
- One numbered question per missing field, in the order listed. Do not add questions.
- Max 18 words per question.
- Ask for the field in the customer's language, not the internal field name.
- Open with one sentence confirming what we are looking into. No greeting, no sign-off.
- Total under 110 words."""

# --- VERSION B: same instruction, plus few-shot examples ---
STEP2_B = STEP2_A + """

EXAMPLES of the style wanted. Match the register and the length, not the content:

  missing field: the order number
  question:      1. What's the order number from your confirmation email?

  missing field: the date the items were dropped off
  question:      2. What day did you drop everything off, roughly?

  missing field: which payout the customer is disputing
  question:      3. Which payout are you asking about, the 1st or the 15th?"""

print('Version B tail:\n', STEP2_B[len(STEP2_A):])

In [ ]:
import re

FILLER = ['we apologize for the inconvenience', 'thank you for reaching out',
          'we are sorry to hear', "we're sorry to hear"]

def score_step2(text, expected_questions):
    numbered = [l for l in text.splitlines() if re.match(r'^\s*\d[\.\)]', l)]
    filler_hits = [f for f in FILLER if f in text.lower()]
    return {
        'questions': len(numbered),
        'q_ok': len(numbered) == expected_questions,
        'words': len(text.split()),
        'len_ok': len(text.split()) < 110,
        'filler': ', '.join(filler_hits) or 'none',
        'filler_ok': not filler_hits,
    }

missing_block = '\n'.join(f'- {m}' for m in step1['missing_info'])
expected = len(step1['missing_info'])

out_a = ask(STEP2_A.format(side=step1['side'], missing=missing_block), system=STEP2_SYSTEM)
out_b = ask(STEP2_B.format(side=step1['side'], missing=missing_block), system=STEP2_SYSTEM)

print('=' * 70)
print('VERSION A - zero-shot')
print('=' * 70)
print(out_a)
print()
print('=' * 70)
print('VERSION B - few-shot with instruction')
print('=' * 70)
print(out_b)

In [ ]:
sa, sb = score_step2(out_a, expected), score_step2(out_b, expected)

print(f'{expected} missing fields, so {expected} questions expected\n')
print(f"{'CHECK':<22}{'A (zero-shot)':>16}{'B (few-shot)':>16}")
print('-' * 54)
for k, label in [('questions', 'questions asked'), ('words', 'word count'), ('filler', 'filler phrases')]:
    print(f'{label:<22}{str(sa[k]):>16}{str(sb[k]):>16}')
print('-' * 54)
pa = sum([sa['q_ok'], sa['len_ok'], sa['filler_ok']])
pb = sum([sb['q_ok'], sb['len_ok'], sb['filler_ok']])
print(f"{'checks passed':<22}{f'{pa}/3':>16}{f'{pb}/3':>16}")

winner = 'B' if pb > pa else 'A' if pa > pb else 'A'
STEP2_WINNER = STEP2_B if winner == 'B' else STEP2_A
step2_out = out_b if winner == 'B' else out_a
print(f"\nWinner: version {winner}" + (' (tie, keeping the simpler prompt)' if pa == pb else ''))
print('The rest of the chain uses the winner.')

### The customer replies

Simulating the inbound answer so the chain runs in one pass. In production this is what the
customer types back.

In [ ]:
CUSTOMER_ANSWERS = """1. marcus.vela@gmail.com
2. dropped off Aug 22 at the 1st Street store
3. 6 items: carhartt detroit jacket, 2 flannels, levis 501s, a starter jacket, a band tee
4. the carhartt - I was told around $200
5. the payout on Sept 1, I got $48 and expected closer to $120"""
print(CUSTOMER_ANSWERS)

## 6. Step 3 - resolve

Four inputs, all produced upstream: the step 1 JSON, the answers step 2's questions triggered, and
the policy lines `retrieve_policy()` selected using step 1's `category`. The retrieved policy is
what keeps it from inventing a refund rule.

In [ ]:
STEP3_SYSTEM = """You are a senior support specialist at VNTG OS. You resolve tickets using only
the policy provided. If the policy does not cover something, you say so and route it to a human
rather than guessing."""

STEP3 = """CLASSIFICATION (from intake):
{classification}

CUSTOMER ANSWERS:
{answers}

POLICY (retrieved for category: {category}):
{policy}

Produce the resolution in exactly this format:

DIAGNOSIS: <2 sentences, cite the specific policy line that applies>
ACTION: <numbered internal steps for the support agent, max 4>
REPLY TO CUSTOMER: <under 120 words, plain English, second person, no promised dollar amount
unless the policy permits it, no apology longer than one clause>
CONFIDENCE: <high | medium | low> - <why, in under 15 words>

Constraints:
- Do not cite a policy rule that is not in the POLICY block above.
- If the POLICY block does not cover the question, say so in DIAGNOSIS and set CONFIDENCE to low.
- If any customer answer conflicts with policy, name the conflict instead of smoothing it over."""

retrieved = retrieve_policy(step1['category'])
print(f"retrieved {len(retrieved.splitlines())} of {TOTAL_LINES} policy lines "
      f"for category '{step1['category']}'\n")

step3_out = ask(
    STEP3.format(
        classification=json.dumps(step1, indent=2),
        answers=CUSTOMER_ANSWERS,
        category=step1['category'],
        policy=retrieved,
    ),
    system=STEP3_SYSTEM,
)
print(step3_out)

## 7. Step 4 - the routing rule

Deliberately **not** a prompt. Routing money to a human is a business rule, so it is Python: it is
auditable, it is testable, and it cannot be talked out of firing by a well-written ticket. The model
only writes the handoff note once the rule has already decided.

In [ ]:
ESCALATION_TRIGGERS = {
    'urgency_high':    lambda c, r, t: c['urgency'] == 'high',
    'payout_over_150': lambda c, r, t: (c.get('payout_at_risk_usd') or 0) > 150,
    'billing_error':   lambda c, r, t: c['category'] == 'billing_error',
    'chargeback_risk': lambda c, r, t: any(k in (t + r).lower()
                                           for k in ['chargeback', 'bank dispute', 'disputing']),
    'low_confidence':  lambda c, r, t: 'CONFIDENCE: low' in r,
}

def escalation_check(classification, resolution, ticket):
    return [name for name, rule in ESCALATION_TRIGGERS.items()
            if rule(classification, resolution, ticket)]

fired_a = escalation_check(step1, step3_out, TICKET_A)
print('Ticket A triggers fired:', fired_a or 'none - agent may send the step 3 reply as written')

In [ ]:
HANDOFF_SYSTEM = 'You write internal handoff notes for support agents. Terse. No customer-facing tone.'

HANDOFF = """This ticket escalated. Triggers: {triggers}

CLASSIFICATION:
{classification}

DRAFT RESOLUTION:
{resolution}

Write the handoff note for the human agent picking this up. Exactly:
WHY ESCALATED: <one line, name the trigger>
WHAT IS KNOWN: <max 3 bullets, facts only>
WHAT THE AGENT MUST DECIDE: <max 2 bullets>
DO NOT SAY TO THE CUSTOMER: <anything the draft promised that policy does not support, or 'nothing'>
Under 120 words total."""

def handoff_note(classification, resolution, triggers):
    return ask(
        HANDOFF.format(triggers=', '.join(triggers),
                       classification=json.dumps(classification, indent=2),
                       resolution=resolution),
        system=HANDOFF_SYSTEM,
    )

if fired_a:
    print(handoff_note(step1, step3_out, fired_a))
else:
    print('No handoff note needed for Ticket A.')

## 8. The whole chain as one function

Same four steps, wrapped, using the A/B winner for step 2. Run against Ticket B, which is written
to trip the routing rule: duplicate charge, a stated bank dispute, and a same-day deadline.

In [ ]:
def run_chain(ticket, simulated_answers, verbose=True):
    trace = {}

    # STEP 1 - classify
    trace['classification'] = json.loads(
        ask(STEP1_V2.format(ticket=ticket), system=STEP1_SYSTEM, json_mode=True)
    )
    cls = trace['classification']

    # STEP 2 - depends on step 1's missing_info
    missing = '\n'.join(f'- {m}' for m in cls['missing_info'])
    trace['questions'] = ask(
        STEP2_WINNER.format(side=cls['side'], missing=missing),
        system=STEP2_SYSTEM,
    )

    # RETRIEVAL - depends on step 1's category
    trace['policy_used'] = retrieve_policy(cls['category'])

    # STEP 3 - depends on steps 1, 2, and the retrieval
    trace['resolution'] = ask(
        STEP3.format(classification=json.dumps(cls, indent=2),
                     answers=simulated_answers,
                     category=cls['category'],
                     policy=trace['policy_used']),
        system=STEP3_SYSTEM,
    )

    # STEP 4 - depends on steps 1 and 3
    trace['triggers'] = escalation_check(cls, trace['resolution'], ticket)
    trace['handoff'] = (handoff_note(cls, trace['resolution'], trace['triggers'])
                        if trace['triggers'] else None)

    if verbose:
        for k, v in trace.items():
            print('=' * 70)
            print(k.upper())
            print('=' * 70)
            print(json.dumps(v, indent=2) if isinstance(v, (dict, list)) else v)
            print()
    return trace

ANSWERS_B = """1. danielle.r@outlook.com
2. order #VN-4417, Tuesday Sept 9
3. both charges show $480.00, 4 minutes apart, Visa ending 1182
4. yes I called Chase this morning"""

trace_b = run_chain(TICKET_B, ANSWERS_B)

In [ ]:
# Side by side: the rule routes the two tickets differently off the same chain,
# and the retrieval hands each one a different slice of policy.
print(f"{'TICKET':<8}{'CATEGORY':<16}{'URGENCY':<9}{'POLICY LINES':<14}{'ROUTED TO'}")
print('-' * 78)
for label, cls, trig in [('A', step1, fired_a),
                         ('B', trace_b['classification'], trace_b['triggers'])]:
    n = len(retrieve_policy(cls['category']).splitlines())
    dest = f"human ({', '.join(trig)})" if trig else 'auto-reply'
    print(f"{label:<8}{cls['category']:<16}{cls['urgency']:<9}{n:<14}{dest}")

## 9. Learning from failed prompts

**Fix 1 - step 1 output format (section 3 vs 4).** v1 returned prose. `json.loads` raised
`JSONDecodeError`, so step 2 had nothing to iterate over. Fixed with a literal key-by-key schema,
enums, and `response_mime_type='application/json'`. This is the change that made it a chain instead
of three prompts in a row.

**Fix 2 - step 2 scope creep.** The first version of step 2 received the raw ticket as well as the
missing fields, and started re-answering the customer's complaint instead of asking questions.
Removing the ticket from that prompt entirely forced the dependency: step 2 can only work from what
step 1 handed it.

**Fix 3 - invented policy in step 3.** With no policy in context the model offered Marcus a
"courtesy credit," which is not a thing VNTG OS does. Passing retrieved policy as context, plus the
negative constraint "do not cite a policy rule that is not in the POLICY block," ended it.

**Fix 4 - passing the entire policy base.** Before the category index, step 3 got the
whole base and started citing shipping rules on a payout ticket. Retrieving by category cut the
context to what applies and removed the cross-talk.

**Fix 5 - routing as code, not a prompt.** The first design asked the model whether to escalate. It
said no on Ticket B roughly one run in three. A dollar threshold and a keyword check are
deterministic, so the same ticket routes the same way every time.

**What the A/B test settled (section 5).** Few-shot versus zero-shot on the same clarifying-question
task, measured rather than eyeballed. Read the table in that cell for this run's result. Across my
runs the zero-shot version drifted on question count when `missing_info` had four or five entries,
and the few-shot version held the one-question-per-field contract. Worth noting the cost: version B
is a longer prompt on every single call, so if the two versions tie, the cheaper prompt wins.